In [1]:
# Cell 1 — Setup & Import
import os
import yaml
import glob
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# Sesuaikan path root project
PROJECT_ROOT = r'C:\futsal-cv'
DATA_YAML = os.path.join(PROJECT_ROOT, 'data', 'data.yaml')

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("Kelas terdaftar:", data_cfg['names'])
print("Jumlah kelas (nc):", data_cfg['nc'])

Error: No connection selected.

In [ ]:
# Cell 2 — Hitung distribusi instance per kelas dari label .txt
def count_instances(label_dir, class_names):
    counter = Counter()
    label_files = glob.glob(os.path.join(label_dir, '*.txt'))
    for lf in label_files:
        with open(lf, 'r') as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    counter[class_names[class_id]] += 1
    return counter

train_labels = os.path.join(PROJECT_ROOT, 'data', 'train', 'labels')
valid_labels = os.path.join(PROJECT_ROOT, 'data', 'valid', 'labels')

train_counts = count_instances(train_labels, data_cfg['names'])
valid_counts = count_instances(valid_labels, data_cfg['names'])

df = pd.DataFrame({'train': train_counts, 'valid': valid_counts}).fillna(0).astype(int)
df['total'] = df['train'] + df['valid']
df = df.sort_values('total', ascending=False)
print(df)

In [ ]:
# Cell 3 — Visualisasi distribusi kelas (langsung kelihatan imbalance-nya)
fig, ax = plt.subplots(figsize=(8, 5))
df['total'].plot(kind='bar', ax=ax, color=['crimson', 'gold', 'orange', 'teal'])
ax.set_title('Distribusi Instance per Kelas — FutsalLens Dataset')
ax.set_ylabel('Jumlah Instance')
ax.set_xlabel('Kelas')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'notebook', 'class_distribution.png'))
plt.show()

In [ ]:
# Cell 4 — Cek rasio train vs valid per kelas (deteksi kalau ada kelas yang minim di valid set)
df['valid_ratio'] = (df['valid'] / df['total'] * 100).round(1)
print(df[['train', 'valid', 'total', 'valid_ratio']])
print("\n⚠️ Kelas dengan valid_ratio terlalu rendah (<10%) berarti mAP-nya kurang reliable secara statistik.")

In [ ]:
# Cell 5 — Visualisasi sample gambar + bounding box (cek kualitas anotasi manual)
import random

def draw_boxes(image_path, label_path, class_names):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    with open(label_path, 'r') as f:
        for line in f:
            cls_id, x, y, bw, bh = map(float, line.split())
            cls_id = int(cls_id)
            x1 = int((x - bw/2) * w)
            y1 = int((y - bh/2) * h)
            x2 = int((x + bw/2) * w)
            y2 = int((y + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, class_names[cls_id], (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

train_images = os.path.join(PROJECT_ROOT, 'data', 'train', 'images')
image_files = glob.glob(os.path.join(train_images, '*.jpg'))
samples = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
    if os.path.exists(label_path):
        img = draw_boxes(img_path, label_path, data_cfg['names'])
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Load model & jalankan evaluasi metrik (mAP per kelas)
from ultralytics import YOLO

MODEL_PATH = r'C:\futsal-cv\runs\detect\runs\futsal\yolov8n_futsal7\weights\best.pt'
model = YOLO(MODEL_PATH)

metrics = model.val(data=DATA_YAML)
print("mAP50 per kelas:")
for i, cls_name in enumerate(data_cfg['names']):
    print(f"  {cls_name}: {metrics.box.maps[i]:.3f}")

In [ ]:
# Cell 7 — Bandingkan mAP per kelas vs jumlah instance (buat justifikasi di laporan)
map_scores = [metrics.box.maps[i] for i in range(len(data_cfg['names']))]
df_compare = pd.DataFrame({
    'class': data_cfg['names'],
    'instance_count': [df.loc[c, 'total'] if c in df.index else 0 for c in data_cfg['names']],
    'mAP50': map_scores
})
print(df_compare)

fig, ax1 = plt.subplots(figsize=(8,5))
ax2 = ax1.twinx()
ax1.bar(df_compare['class'], df_compare['instance_count'], color='steelblue', alpha=0.6, label='Instance Count')
ax2.plot(df_compare['class'], df_compare['mAP50'], color='crimson', marker='o', label='mAP50')
ax1.set_ylabel('Instance Count')
ax2.set_ylabel('mAP50')
plt.title('Korelasi Jumlah Instance vs Performa Model per Kelas')
plt.tight_layout()
plt.show()